# 02 Retrieval Experiments

Notebook này là nơi thử nghiệm retrieval trước khi chọn pipeline cho app/deploy.

Các method được so sánh trực tiếp trong notebook: `bm25`, `vector`, `hybrid`, `hybrid_rrf`, `hybrid_rrf_mmr`.
Source production trong `src/retrieval/pipeline.py` chỉ giữ method được chọn sau benchmark.

Mặc định `USE_TFIDF_FALLBACK=true` để smoke test offline nhanh. Set `USE_TFIDF_FALLBACK=false` trước khi chạy notebook để benchmark multilingual sentence embeddings + FAISS cho report production.

In [ ]:
import json
import os
from collections import defaultdict
from math import log2
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import faiss
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer

from src.data.chunking import chunk_documents
from src.data.clean_text import normalize_for_match, tokenize_vi
from src.data.load_dataset import load_documents_from_jsonl
from src.data.schema import LegalChunk, SearchResult

CORPUS_PATH = PROJECT_ROOT / "data/processed/labor_corpus.jsonl"
REPORT_JSON = PROJECT_ROOT / "reports/retrieval_evaluation.json"
REPORT_MD = PROJECT_ROOT / "reports/evaluation.md"
USE_TFIDF_FALLBACK = os.getenv("USE_TFIDF_FALLBACK", "true").lower() == "true"

In [ ]:
TEST_QUERIES = [
    {"question": "Người lao động đơn phương chấm dứt hợp đồng lao động cần báo trước bao lâu?", "terms": ["hợp đồng lao động", "người lao động", "đơn phương chấm dứt"]},
    {"question": "Trường hợp nào người lao động được nhận trợ cấp thôi việc?", "terms": ["trợ cấp thôi việc", "người lao động"]},
    {"question": "Doanh nghiệp có trách nhiệm gì về an toàn vệ sinh lao động?", "terms": ["an toàn vệ sinh lao động", "người sử dụng lao động"]},
    {"question": "Quy định về tiền lương và lương tối thiểu của người lao động là gì?", "terms": ["tiền lương", "lương tối thiểu", "người lao động"]},
    {"question": "Người lao động nước ngoài cần điều kiện gì để làm việc tại Việt Nam?", "terms": ["lao động nước ngoài", "giấy phép lao động"]},
    {"question": "Kỷ luật sa thải người lao động được áp dụng trong trường hợp nào?", "terms": ["kỷ luật lao động", "sa thải", "người lao động"]},
    {"question": "Bảo hiểm thất nghiệp hỗ trợ người lao động như thế nào?", "terms": ["bảo hiểm thất nghiệp", "người lao động"]},
    {"question": "Tranh chấp lao động tập thể được giải quyết như thế nào?", "terms": ["tranh chấp lao động", "tập thể"]},
]

documents = load_documents_from_jsonl(str(CORPUS_PATH))
chunks = chunk_documents(documents)
print(f"documents={len(documents)}, chunks={len(chunks)}")

In [ ]:
def chunk_text(chunk: LegalChunk) -> str:
    return f"{chunk.title} {chunk.text}"


def normalize_rows(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-12
    return np.asarray(matrix / norms, dtype="float32")


bm25 = BM25Okapi([tokenize_vi(chunk_text(chunk)) for chunk in chunks])
passages = [f"passage: {chunk_text(chunk)}" for chunk in chunks]
if USE_TFIDF_FALLBACK:
    vector_backend = "tfidf"
    vectorizer = TfidfVectorizer(max_features=384, ngram_range=(1, 2))
    embeddings = normalize_rows(vectorizer.fit_transform(passages).toarray())

    def encode_query(query: str) -> np.ndarray:
        return normalize_rows(vectorizer.transform([query]).toarray())
else:
    from sentence_transformers import SentenceTransformer

    vector_backend = "sentence_transformer"
    encoder = SentenceTransformer("intfloat/multilingual-e5-small")
    embeddings = np.asarray(encoder.encode(passages, normalize_embeddings=True, show_progress_bar=True), dtype="float32")

    def encode_query(query: str) -> np.ndarray:
        return np.asarray(encoder.encode([f"query: {query}"], normalize_embeddings=True), dtype="float32")

print("vector backend:", vector_backend)
vector_index = faiss.IndexFlatIP(embeddings.shape[1])
vector_index.add(embeddings)


def retrieve_bm25(query: str, top_k: int) -> list[SearchResult]:
    scores = bm25.get_scores(tokenize_vi(query))
    indices = sorted(range(len(scores)), key=lambda idx: scores[idx], reverse=True)[:top_k]
    return [SearchResult(chunks[idx], float(scores[idx]), "bm25", rank) for rank, idx in enumerate(indices, start=1)]


def retrieve_vector(query: str, top_k: int) -> list[SearchResult]:
    query_vector = encode_query(query)
    scores, indices = vector_index.search(query_vector, top_k)
    return [SearchResult(chunks[int(idx)], float(score), "vector", rank) for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1) if idx >= 0]


def minmax(results: list[SearchResult]) -> dict[str, float]:
    scores = [result.score for result in results]
    low, high = min(scores), max(scores)
    if high == low:
        return {result.chunk.chunk_id: 1.0 for result in results}
    return {result.chunk.chunk_id: (result.score - low) / (high - low) for result in results}


def weighted_hybrid(sparse: list[SearchResult], vector: list[SearchResult], top_k: int, alpha: float = 0.55) -> list[SearchResult]:
    by_id = {result.chunk.chunk_id: result.chunk for result in sparse + vector}
    sparse_scores, vector_scores = minmax(sparse), minmax(vector)
    scores = {chunk_id: (1 - alpha) * sparse_scores.get(chunk_id, 0.0) + alpha * vector_scores.get(chunk_id, 0.0) for chunk_id in by_id}
    ordered = sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_k]
    return [SearchResult(by_id[chunk_id], float(score), "hybrid", rank) for rank, (chunk_id, score) in enumerate(ordered, start=1)]


def rrf(result_lists: list[list[SearchResult]], top_k: int) -> list[SearchResult]:
    scores, by_id = defaultdict(float), {}
    for results in result_lists:
        for rank, result in enumerate(results, start=1):
            by_id[result.chunk.chunk_id] = result.chunk
            scores[result.chunk.chunk_id] += 1.0 / (60 + rank)
    ordered = sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_k]
    return [SearchResult(by_id[chunk_id], float(score), "hybrid_rrf", rank) for rank, (chunk_id, score) in enumerate(ordered, start=1)]


def mmr(results: list[SearchResult], top_k: int, lambda_mult: float = 0.7) -> list[SearchResult]:
    if len(results) <= top_k:
        return results
    vectors = normalize_rows(TfidfVectorizer(max_features=512).fit_transform([chunk_text(result.chunk) for result in results]).toarray())
    relevance = np.asarray([result.score for result in results], dtype="float32")
    relevance = (relevance - relevance.min()) / (relevance.max() - relevance.min() + 1e-12)
    selected = [int(np.argmax(relevance))]
    remaining = set(range(len(results))) - set(selected)
    while remaining and len(selected) < top_k:
        best_idx = max(remaining, key=lambda idx: lambda_mult * float(relevance[idx]) - (1 - lambda_mult) * max(float(vectors[idx] @ vectors[chosen]) for chosen in selected))
        selected.append(best_idx)
        remaining.remove(best_idx)
    return [SearchResult(results[idx].chunk, results[idx].score, "hybrid_rrf_mmr", rank) for rank, idx in enumerate(selected, start=1)]


def retrieve(query: str, method: str, top_k: int = 8) -> list[SearchResult]:
    candidate_k = min(len(chunks), max(30, top_k * 5))
    sparse, vector = retrieve_bm25(query, candidate_k), retrieve_vector(query, candidate_k)
    if method == "bm25":
        return sparse[:top_k]
    if method == "vector":
        return vector[:top_k]
    if method == "hybrid":
        return weighted_hybrid(sparse, vector, top_k)
    fused = rrf([sparse, vector], candidate_k)
    return mmr(fused, top_k) if method == "hybrid_rrf_mmr" else fused[:top_k]

In [ ]:
def is_relevant(result: SearchResult, terms: list[str]) -> bool:
    text = normalize_for_match(chunk_text(result.chunk))
    return any(term in text for term in terms)


def metrics(results: list[SearchResult], terms: list[str]) -> dict[str, float]:
    gains = [1.0 if is_relevant(result, terms) else 0.0 for result in results[:5]]
    dcg = sum(gain / log2(idx + 2) for idx, gain in enumerate(gains))
    ideal_dcg = sum(gain / log2(idx + 2) for idx, gain in enumerate(sorted(gains, reverse=True)))
    first_relevant = next((idx for idx, result in enumerate(results, start=1) if is_relevant(result, terms)), None)
    return {
        "recall@5": 1.0 if any(gains) else 0.0,
        "mrr": 1.0 / first_relevant if first_relevant else 0.0,
        "ndcg@5": dcg / ideal_dcg if ideal_dcg else 0.0,
        "citation_coverage": sum(bool(result.chunk.title and result.chunk.doc_id) for result in results) / max(len(results), 1),
    }


METHODS = ["bm25", "vector", "hybrid", "hybrid_rrf", "hybrid_rrf_mmr"]
report = {"num_documents": len(documents), "num_chunks": len(chunks), "vector_backend": vector_backend, "methods": {}}
for method in METHODS:
    details = []
    for item in TEST_QUERIES:
        results = retrieve(item["question"], method)
        details.append({"question": item["question"], **metrics(results, item["terms"]), "top_title": results[0].chunk.title if results else None})
    report["methods"][method] = {key: sum(row[key] for row in details) / len(details) for key in ["recall@5", "mrr", "ndcg@5", "citation_coverage"]}
    report["methods"][method]["details"] = details

rows = [{"method": method, **{key: value for key, value in values.items() if key != "details"}} for method, values in report["methods"].items()]
df = pd.DataFrame(rows)
df["balanced_score"] = 0.45 * df["recall@5"] + 0.35 * df["mrr"] + 0.20 * df["ndcg@5"]
df.sort_values("balanced_score", ascending=False)

In [ ]:
selected_method = df.sort_values("balanced_score", ascending=False).iloc[0]["method"]
assert selected_method == "hybrid_rrf", f"Re-check production pipeline: benchmark selected {selected_method}"

if USE_TFIDF_FALLBACK:
    print("TF-IDF smoke mode: production report was not overwritten.")
else:
    REPORT_JSON.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    lines = [
        "# Retrieval Evaluation", "", f"- Documents: {len(documents)}", f"- Chunks: {len(chunks)}", f"- Vector backend: `{vector_backend}`", "",
        "| Method | Recall@5 | MRR | nDCG@5 | Citation coverage | Balanced score |",
        "|---|---:|---:|---:|---:|---:|",
    ]
    for row in df.to_dict("records"):
        lines.append(f"| {row['method']} | {row['recall@5']:.3f} | {row['mrr']:.3f} | {row['ndcg@5']:.3f} | {row['citation_coverage']:.3f} | {row['balanced_score']:.3f} |")
    lines.extend(["", f"Recommended balanced method: `{selected_method}`.", ""])
    REPORT_MD.write_text("\n".join(lines), encoding="utf-8")
    print("Updated production retrieval report.")
print(f"Selected method: {selected_method}")

## Selected method for app/deploy

Pipeline production giữ **`hybrid_rrf`** vì benchmark cân bằng tốt và thiết kế dễ giải thích:
- BM25 xử lý tốt thuật ngữ pháp lý chính xác.
- FAISS vector retrieval xử lý câu hỏi tự nhiên.
- RRF tránh phụ thuộc vào scale điểm khác nhau giữa sparse và vector retrieval.

MMR và weighted fusion chỉ nằm trong notebook để so sánh, không nằm trong source deploy.